# Model tournament: Kaggle proof run (2 x T4)
Purpose: prove that every candidate architecture builds, trains, stops/resumes and produces a report **before any DGX time is spent**.
This is a machinery test (3M tokens per model), not a scientific result.

**Before you run (Notebook settings):** Accelerator **GPU T4 x2**, Internet **on**, add the folder `qwenlab` (v0.4) as a dataset input,
Add-ons > Secrets: attach `HF_TOKEN` (write access).

**Overnight run:** click **Save Version > Save & Run All (Commit)** (not the plain Run button). Kaggle then runs the whole notebook in the background for up to 12 hours,
even with the browser closed, and keeps `/kaggle/working` as the notebook's Output. Next morning open the saved version: **Logs** shows every cell's output, **Output** has `tournament/` (report, runs) and the plan.
Expected total time 2-4 hours. To run it interactively instead, run the cells one by one and check each "Pass when" note.

In [ ]:
# ---------- 1. SETTINGS (edit these) ----------
TOKENIZER_PATH   = ""        # e.g. "/kaggle/input/malayalam-tokenizer/malayalam_unigram32k.model". Leave "" if the tokenizer is not ready:
                             # then only the check + smoke phases run (they do not need it).
HF_REPO          = "Akzhai/malayalam-lm-tournament"   # one repo for everything (private)
SUBDIR           = "kaggle_A"                          # second Kaggle account: "kaggle_B" so uploads never overwrite each other
BACKUP_CHECKPOINTS = False   # True: every run also backs its checkpoints up to HF_REPO (6 GB per upload; not needed for this 1-hour proof)

In [ ]:
# ---------- 2. SETUP: copy the code (read-only input), install, load the HF token ----------
import os, shutil, subprocess, json
found = [p for p in subprocess.run(['find','/kaggle/input','-iname','pyproject.toml'], capture_output=True, text=True).stdout.split() if p]
assert found, 'No pyproject.toml under /kaggle/input: add the qwenlab folder as a dataset input (Add Input) and run again.'
SRC = found[0].rsplit('/', 1)[0]
PKG = '/kaggle/working/qwenlab_pkg'
shutil.copytree(SRC, PKG, dirs_exist_ok=True)
os.chdir(PKG)
print('code at', PKG)
!pip install -q -e "{PKG}[hf-archs]" sentencepiece pyarrow
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    print('NO HF_TOKEN secret:', type(e).__name__, '(uploads will be skipped)')
import torch, transformers
!df -h /kaggle/working | tail -1
print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPUs', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# ---------- 3. Build the run plan from the smoke plan + your settings (no JSON editing by hand) ----------
HAVE_TOKENIZER = bool(TOKENIZER_PATH) and os.path.exists(TOKENIZER_PATH)
plan = json.load(open('configs/tournament/smoke_kaggle.json', encoding='utf-8'))
if HAVE_TOKENIZER:
    plan['overrides'] += [f'tokenizer.path="{TOKENIZER_PATH}"']
if BACKUP_CHECKPOINTS:
    plan['hub_repo'] = HF_REPO
PLAN_RUN = '/kaggle/working/plan_run.json'
json.dump(plan, open(PLAN_RUN, 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
VOCAB_ARG = '' if HAVE_TOKENIZER else '--vocab 32000'
print('tokenizer ready:', HAVE_TOKENIZER, '| candidates:', [c['id'] for c in plan['candidates']])

## 4. Unit tests  -  *Pass when:* every test passes (the new torch tests run here for the first time; send me the failure text if any fail)

In [ ]:
!cd "$PKG" && python -m pytest tests -q -x

## 5. Check: every candidate at FULL size on CPU
*Pass when:* each line shows `status: ok`, `params_M` about 385-420, `loss` close to `ln_vocab` (about 10.4), and the `[parity]` line says MATCH.
A `FAILED:` line names the exact problem (usually a transformers version without that architecture). Fix it or remove that candidate; do not carry it to the DGX.

In [ ]:
!cd "$PKG" && python scripts/tournament.py --plan $PLAN_RUN --phase check $VOCAB_ARG

## 6. Smoke: every candidate at TINY size, loss must fall, stop + resume must work  -  *Pass when:* `SMOKE OK`

In [ ]:
!cd "$PKG" && python scripts/tournament.py --plan $PLAN_RUN --phase smoke

## 7-10. Real-size phases (need the tokenizer + the Malayalam dataset in `configs/malayalam.json`)
If `HAVE_TOKENIZER` is False these cells do nothing; finish the tokenizer, set `TOKENIZER_PATH`, and re-run from cell 3.

In [ ]:
# 7. build the shared tokenised cache once (20M tokens for the smoke)
if HAVE_TOKENIZER:
    !cd "$PKG" && python scripts/tournament.py --plan $PLAN_RUN --phase prepare
else:
    print('skipped: no tokenizer yet')

In [ ]:
# 8. probe: tokens/s, peak memory, best micro-batch, longest context per candidate (GPU 0). T4 numbers are small; only check that nothing crashes.
if HAVE_TOKENIZER:
    !cd "$PKG" && python scripts/tournament.py --plan $PLAN_RUN --phase probe --gpus 0
else:
    print('skipped: no tokenizer yet')

In [ ]:
# 9. race: 9 candidates x ~3M tokens on both T4s (about 1 hour). Re-running this cell skips finished runs and resumes unfinished ones.
if HAVE_TOKENIZER:
    !cd "$PKG" && python scripts/tournament.py --plan $PLAN_RUN --phase race --gpus 0,1
else:
    print('skipped: no tokenizer yet')

In [ ]:
# 10. report (a table that lists every candidate = the machinery works; the ranking itself means nothing at 3M tokens)
if HAVE_TOKENIZER:
    !cd "$PKG" && python scripts/tournament.py --plan $PLAN_RUN --phase report
else:
    print('skipped: no tokenizer yet')

## 11. Upload the findings to Hugging Face (report, probe, run metrics/configs/logs; no checkpoints)

In [ ]:
!cd "$PKG" && python scripts/upload_results.py --plan $PLAN_RUN --repo $HF_REPO --subdir $SUBDIR

## Done. What to do next
* All phases passed -> go to the DGX (`docs/DGX_SETUP.md`, then `dgx/tournament_dgx.ipynb`).
* Something failed -> copy the failing cell's output; most likely fixes are a transformers version or one candidate to drop.
* Interrupted session? Re-run cells 1-3, then the phase that stopped: finished work is skipped, unfinished runs resume from the last checkpoint.